In [1]:
import pandas as pd
import numpy as np

In [2]:
PATH = '../data/'
df = pd.read_csv(PATH + 'msmes.csv')
df.head()

,Business_Size,Business_Age,Industry_Type,Entity_Type,Monthly_GST_Sales,GST_Growth_Rate,GST_Filing_Delay,GST_Compliance_Rate,Monthly_UPI_Count,Monthly_UPI_Value,...,Monthly_Credit,Monthly_Debit,Cashflow_Volatility,Employee_Count,Payroll_Consistency,Vendor_Payment_Delay,Working_Capital_Cycle,EMI_Bounce_Count,PD_True,Default
0,Small,3.259143,Retail,Proprietorship,1.041611e+06,-0.105973,11.100083,0.930616,89,222313.252027,...,1.079347e+06,799912.170455,0.016356,27,0.975329,11.376241,34.511462,0,0.000059,0
1,Micro,1.043744,Manufacturing,LLP,1.185538e+05,0.148953,8.625211,1.000000,13,5981.733679,...,1.101660e+05,92536.163891,0.341500,7,0.849837,19.329942,102.988904,0,0.001164,0
2,Micro,7.300253,Agriculture,Proprietorship,5.511186e+05,0.110490,4.305277,0.959657,17,17563.091000,...,4.778285e+05,279259.848766,0.527937,4,0.799129,20.347545,78.429116,1,0.103713,1
3,Micro,26.798767,Retail,Proprietorship,1.669279e+05,-0.015217,3.290730,1.000000,59,80650.048906,...,1.557148e+05,130123.809195,0.382392,5,0.706147,7.669705,43.484573,0,0.000008,0
4,Micro,6.507175,Manufacturing,Partnership,8.990894e+05,0.127848,3.903448,0.931603,14,31292.801883,...,9.385547e+05,755815.166491,0.174035,5,0.707897,19.509115,103.379616,1,0.007137,0


## Check the overall default rate

In [3]:
print(df['Default'].value_counts())
print(df['Default'].mean())

Default
0    17862
1     2138
Name: count, dtype: int64
0.1069


## Verify PD calibration

In [4]:
bins = pd.qcut(df["PD_True"], 10)

calibration = (
    df.groupby(bins)
      .agg(
          Avg_PD=("PD_True","mean"),
          Observed_Default=("Default","mean"),
          Count=("Default","size")
      )
)

print(calibration)

                                   Avg_PD  Observed_Default  Count
PD_True                                                           
(-0.0009999999999761, 5.74e-05]  0.000023            0.0000   2000
(5.74e-05, 0.000218]             0.000125            0.0000   2000
(0.000218, 0.000589]             0.000381            0.0005   2000
(0.000589, 0.00146]              0.000968            0.0000   2000
(0.00146, 0.00349]               0.002339            0.0055   2000
(0.00349, 0.00919]               0.005849            0.0070   2000
(0.00919, 0.0261]                0.015902            0.0155   2000
(0.0261, 0.0903]                 0.050781            0.0450   2000
(0.0903, 0.442]                  0.223214            0.2270   2000
(0.442, 1.0]                     0.775613            0.7685   2000


## Verify monotonic relationships

In [5]:
features = [
    "GST_Filing_Delay",
    "GST_Compliance_Rate",
    "Cashflow_Volatility",
    "EMI_Bounce_Count",
    "Vendor_Payment_Delay",
    "Business_Age",
    "Average_Bank_Balance",
    "Digital_Sales_Ratio",
    "GST_Growth_Rate"
]

for feature in features:
    print(f"\n{'='*60}")
    print(feature)

    monotonic = (
        df.groupby(pd.qcut(df[feature], q=5, duplicates="drop"))
          ["Default"]
          .mean()
    )

    print(monotonic)


GST_Filing_Delay
GST_Filing_Delay
(0.0744, 3.8]        0.01900
(3.8, 6.529]         0.03400
(6.529, 9.983]       0.05425
(9.983, 15.547]      0.09850
(15.547, 120.473]    0.32875
Name: Default, dtype: float64

GST_Compliance_Rate
GST_Compliance_Rate
(0.331, 0.89]     0.32700
(0.89, 0.931]     0.10075
(0.931, 0.963]    0.05325
(0.963, 0.998]    0.03100
(0.998, 1.0]      0.02250
Name: Default, dtype: float64

Cashflow_Volatility
Cashflow_Volatility
(0.00118, 0.181]    0.02750
(0.181, 0.272]      0.05175
(0.272, 0.361]      0.06575
(0.361, 0.481]      0.11825
(0.481, 1.0]        0.27125
Name: Default, dtype: float64

EMI_Bounce_Count
EMI_Bounce_Count
(-0.001, 1.0]    0.068391
(1.0, 2.0]       0.184433
(2.0, 5.0]       0.376895
Name: Default, dtype: float64

Vendor_Payment_Delay
Vendor_Payment_Delay
(3.241, 8.715]      0.03375
(8.715, 11.409]     0.04250
(11.409, 14.576]    0.08150
(14.576, 18.549]    0.14750
(18.549, 50.19]     0.22925
Name: Default, dtype: float64

Business_Age
Business

## Compare risky vs safe businesses

In [6]:
safe = df[df.Default==0]
risky = df[df.Default==1]
pd.concat([safe.mean(numeric_only=True), risky.mean(numeric_only=True)], axis = 1).set_axis(['safe', 'risksy'], axis = 1).round(3)

,safe,risksy
Business_Age,6.265,4.119
Monthly_GST_Sales,740604.479,649257.243
GST_Growth_Rate,0.121,0.125
GST_Filing_Delay,9.097,21.583
GST_Compliance_Rate,0.944,0.862
Monthly_UPI_Count,36.926,23.331
Monthly_UPI_Value,52456.884,33123.167
Digital_Sales_Ratio,0.703,0.546
Average_Bank_Balance,109529.359,71661.354
Monthly_Credit,703738.083,615322.850
